# jlens lab — Colab backend

Serves the interactive Jacobian-lens lab (generate → click token → per-layer readout)
for the **base / dark / clinical-depression** organisms, tunneled via cloudflared.

Run all cells; the last one prints a public `trycloudflare.com` URL — open it, that's the lab.

Needs a GPU runtime (A100 recommended; one 8B model on GPU at a time, switching takes ~1-2 min).

In [ ]:
import os
if not os.path.exists("dt_rl") and not os.path.basename(os.getcwd()) == "dt_rl":
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
!git pull
%run notebooks/colab_setup.py

In [ ]:
%pip install -q -U transformers accelerate sentencepiece huggingface_hub fastapi uvicorn
%pip install -q -e third_party/jacobian-lens
import jlens; print("jlens imported:", jlens.__file__)

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception: pass
print("HF_TOKEN:", "set" if os.environ.get("HF_TOKEN") else "not set")

In [ ]:
# cloudflared (no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# launch server (preloads base) + tunnel, print the public URL
import re, subprocess, sys, threading, time

PORT = 8000

# kill leftovers from a previous run of this cell
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
time.sleep(2)

server = subprocess.Popen(
    [sys.executable, "scripts/lens_lab_server.py", "--port", str(PORT), "--preload", "base"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
threading.Thread(target=lambda: [print("[server]", l, end="") for l in server.stdout], daemon=True).start()

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

print("\n" + "=" * 60)
print(f"  LAB URL: {url}")
print("=" * 60)
print("(server keeps loading the base model in the background;")
print(" the page is usable once /api/config responds — give it ~2 min)")

In [ ]:
# optional: keep the runtime alive / watch server logs
import time
while True:
    time.sleep(300)
    print("alive", time.strftime("%H:%M:%S"))